# Day 2 Lab — Advanced Python
## Iterators, Generators, Decorators & Context Managers — Hands-On Lab

This lab is a large, standalone set of exercises to reinforce Day 2's Advanced Python topics. Work through each part in order; solutions are provided below each task.

### Structure
- Part A: Iterators (Exercises 1–3)
- Part B: Generators (Exercises 4–7)
- Part C: Decorators (Exercises 8–12)
- Part D: Context Managers (Exercises 13–16)
- Part E: Mini Challenges — Combine Everything (Exercises 17–20)


## Part A — Iterators

### Exercise 1: `EvenNumbers` Iterator
Build a custom iterator class `EvenNumbers` that iterates over even numbers from 0 up to (and including) a given limit.


In [1]:
# Your solution here

class EvenNumbers:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration
        val = self.current
        self.current += 2
        return val

print(list(EvenNumbers(10)))

[0, 2, 4, 6, 8, 10]


### Exercise 2: `CyclicIterator`
Build an iterator that cycles through a list forever (careful: infinite!). Then use `itertools.islice` (or a manual counter) to only take the first 7 values so it doesn't run forever.


In [2]:
# Your solution here

import itertools

class CyclicIterator:
    def __init__(self, items):
        if not items:
            raise ValueError("Sequence cannot be empty.")
        self.items = list(items)
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        val = self.items[self.index]
        self.index = (self.index + 1) % len(self.items)
        return val

cycle = CyclicIterator(['A', 'B', 'C'])
print(list(itertools.islice(cycle, 7)))

['A', 'B', 'C', 'A', 'B', 'C', 'A']


### Exercise 3: Make a Custom Class Iterable (Without Being an Iterator Itself)
Build a `WeekDays` class whose `__iter__` method returns a **separate iterator object**, so multiple independent loops over the same `WeekDays` instance don't interfere with each other.


In [3]:
# Your solution here

class WeekDaysIterator:
    def __init__(self, days):
        self.days = days
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.days):
            raise StopIteration
        day = self.days[self.index]
        self.index += 1
        return day

class WeekDays:
    def __init__(self):
        self.days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    def __iter__(self):
        return WeekDaysIterator(self.days)

week = WeekDays()
for d1 in week:
    for d2 in week:
        pass
print("Independent iterations verified.")

Independent iterations verified.


## Part B — Generators

### Exercise 4: Fibonacci Generator
Write a generator function `fibonacci(n)` that yields the first `n` Fibonacci numbers, without storing them all in a list first.


In [4]:
# Your solution here

def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

print(list(fibonacci(10)))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Exercise 5: Infinite Generator With a Stop Condition
Write an infinite generator `natural_numbers()` that yields 1, 2, 3, ... forever. Then write code that consumes it but stops once it finds the first number greater than 1000 that is divisible by 7 and 11.


In [5]:
# Your solution here

def natural_numbers():
    n = 1
    while True:
        yield n
        n += 1

target = None
for num in natural_numbers():
    if num > 1000 and num % 7 == 0 and num % 11 == 0:
        target = num
        break

print(f"Target number: {target}")

Target number: 1001


### Exercise 6: File-Line Style Generator (Simulated)
Simulate reading a huge log file line by line using a generator, then write a generator pipeline: one generator yields raw lines, a second generator filters only lines containing the word `"ERROR"`.


In [6]:
# Your solution here

def raw_log_reader():
    simulated_logs = [
        "2026-08-30 INFO Service started",
        "2026-08-30 DEBUG Cache initialized",
        "2026-08-30 ERROR Database timeout on port 5432",
        "2026-08-30 INFO Request handled in 12ms",
        "2026-08-30 ERROR Disk usage exceeded 90%",
    ]
    for line in simulated_logs:
        yield line

def filter_errors(lines):
    for line in lines:
        if "ERROR" in line:
            yield line

for error in filter_errors(raw_log_reader()):
    print(error)

2026-08-30 ERROR Database timeout on port 5432
2026-08-30 ERROR Disk usage exceeded 90%


### Exercise 7: Generator Expression for Data Processing
Given a list of dictionaries representing sensor readings, use a generator expression (not a list comprehension) to compute the average temperature without building an intermediate list.


In [7]:
# Your solution here

readings = [
    {"sensor": "A", "temp": 21.5},
    {"sensor": "B", "temp": 23.0},
    {"sensor": "C", "temp": 20.8},
    {"sensor": "D", "temp": 22.7},
]

total_readings = len(readings)
avg_temp = sum(item["temp"] for item in readings) / total_readings if total_readings else 0
print(f"Average Temperature: {avg_temp:.2f}")

Average Temperature: 22.00


## Part C — Decorators

### Exercise 8: `uppercase_result` Decorator
Write a decorator `uppercase_result` that converts a function's string return value to uppercase.


In [8]:
# Your solution here

from functools import wraps

def uppercase_result(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if isinstance(result, str):
            return result.upper()
        return result
    return wrapper

@uppercase_result
def greet(name):
    return f"hello, {name}"

print(greet("alice"))

HELLO, ALICE


### Exercise 9: `retry` Decorator
Write a decorator `retry(max_attempts)` (a decorator factory) that retries the decorated function up to `max_attempts` times if it raises an exception, printing which attempt failed, and finally re-raising if all attempts fail.


In [9]:
# Your solution here

from functools import wraps

def retry(max_attempts):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

attempts_counter = 0

@retry(max_attempts=3)
def unstable_call():
    global attempts_counter
    attempts_counter += 1
    if attempts_counter < 3:
        raise ValueError("Network glitch")
    return "Success!"

print(unstable_call())

Attempt 1/3 failed: Network glitch
Attempt 2/3 failed: Network glitch
Success!


### Exercise 10: Authentication / Access Control Decorator
Simulate a simple permission system: write a decorator `require_role(role)` that only allows a function to run if a global `current_user` dict has a matching `"role"` key; otherwise it prints an "Access Denied" message and does not call the function.


In [10]:
# Your solution here

from functools import wraps

current_user = {"username": "medhat", "role": "user"}

def require_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") == role:
                return func(*args, **kwargs)
            print(f"Access Denied: Requires role '{role}'.")
            return None
        return wrapper
    return decorator

@require_role("admin")
def delete_records():
    return "Records deleted."

delete_records()

Access Denied: Requires role 'admin'.


### Exercise 11: Caching Decorator (Manual, Without `functools`)
Write your own caching decorator `simple_cache` from scratch (without using `functools.lru_cache`) that stores results in a dictionary keyed by the function's arguments.


In [11]:
# Your solution here

def simple_cache(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = (args, frozenset(kwargs.items()))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    return wrapper

@simple_cache
def expensive_add(a, b):
    print("Computing...")
    return a + b

print(expensive_add(2, 3))
print(expensive_add(2, 3))

Computing...
5
5


### Exercise 12: Logging Decorator That Preserves Metadata
Write a decorator `logged` that prints the function name, arguments, and return value every time it's called — and correctly preserves the original function's `__name__` and docstring using `functools.wraps`.


In [12]:
# Your solution here

from functools import wraps

def logged(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        arg_str = ", ".join([str(a) for a in args] + [f"{k}={v}" for k, v in kwargs.items()])
        result = func(*args, **kwargs)
        print(f"Called {func.__name__}({arg_str}) -> {result}")
        return result
    return wrapper

@logged
def multiply(x, y):
    """Multiplies two numbers."""
    return x * y

multiply(4, 5)
print(f"Name: {multiply.__name__}, Doc: {multiply.__doc__}")

Called multiply(4, 5) -> 20
Name: multiply, Doc: Multiplies two numbers.


## Part D — Context Managers

### Exercise 13: `Timer` Context Manager With Exception Safety
Write a class-based context manager `Timer` that prints elapsed time on exit **even if an exception occurs** inside the `with` block (but doesn't suppress the exception).


In [13]:
# Your solution here

import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.perf_counter() - self.start
        print(f"Elapsed time: {elapsed:.6f} seconds")
        return False  # Propagate exception if one occurred

try:
    with Timer():
        time.sleep(0.01)
        raise RuntimeError("Something failed")
except RuntimeError:
    print("Caught expected exception.")

Elapsed time: 0.010154 seconds
Caught expected exception.


### Exercise 14: `open_file_safe` Context Manager
Write a context manager class `open_file_safe` that mimics `open()` — it should open a file on `__enter__` and guarantee it's closed on `__exit__`, printing a confirmation message when the file is closed.


In [14]:
# Your solution here

class open_file_safe:
    def __init__(self, filename, mode='r', **kwargs):
        self.filename = filename
        self.mode = mode
        self.kwargs = kwargs
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, self.mode, **self.kwargs)
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
            print(f"File '{self.filename}' closed safely.")
        return False

with open_file_safe("demo.txt", "w") as f:
    f.write("Hello context manager!")

File 'demo.txt' closed safely.


### Exercise 15: `contextlib`-Based Database Connection Simulator
Using `@contextmanager` from `contextlib`, write a `db_connection(name)` context manager that prints "Connecting to <name>" on entry and "Closing connection to <name>" on exit, even if an error happens inside.


In [15]:
# Your solution here

from contextlib import contextmanager

@contextmanager
def db_connection(name):
    print(f"Connecting to {name}")
    try:
        yield name
    finally:
        print(f"Closing connection to {name}")

with db_connection("PostgreSQL_Prod") as conn:
    print(f"Executing query on {conn}...")

Connecting to PostgreSQL_Prod
Executing query on PostgreSQL_Prod...
Closing connection to PostgreSQL_Prod


### Exercise 16: Nested Context Managers
Use two context managers together in a single `with` statement (e.g., two simulated resources) and observe the order in which `__enter__` and `__exit__` are called.


In [16]:
# Your solution here

from contextlib import contextmanager

@contextmanager
def resource(name):
    print(f"Entering {name}")
    try:
        yield name
    finally:
        print(f"Exiting {name}")

# Test enter and exit ordering
with resource("Resource A") as a, resource("Resource B") as b:
    print(f"Working with {a} and {b}")

Entering Resource A
Entering Resource B
Working with Resource A and Resource B
Exiting Resource B
Exiting Resource A


## Part E — Mini Challenges (Combine Everything)

### Exercise 17: Lazy Data Pipeline
Build a data-processing pipeline using **only generators**:
1. `read_numbers()` — yields numbers 1 to 50
2. `filter_multiples_of_3(numbers)` — yields only multiples of 3
3. `square(numbers)` — yields each number squared

Chain them together and print the final results.


In [17]:
# Your solution here

def read_numbers():
    for i in range(1, 51):
        yield i

def filter_multiples_of_3(numbers):
    for n in numbers:
        if n % 3 == 0:
            yield n

def square(numbers):
    for n in numbers:
        yield n * n

pipeline = square(filter_multiples_of_3(read_numbers()))
print(list(pipeline))

[9, 36, 81, 144, 225, 324, 441, 576, 729, 900, 1089, 1296, 1521, 1764, 2025, 2304]


### Exercise 18: Timing + Retry Decorator Stack
Combine the `timer` and `retry` decorators from today so a function is both timed AND retried on failure. Apply both decorators to a function that randomly fails.


In [18]:
# Your solution here

import time
import random
from functools import wraps

def timer_dec(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        res = func(*args, **kwargs)
        t1 = time.perf_counter()
        print(f"[{func.__name__}] Execution took {t1 - t0:.4f}s")
        return res
    return wrapper

def retry_dec(max_attempts):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {i}/{max_attempts} failed: {e}")
                    if i == max_attempts:
                        raise
        return wrapper
    return decorator

@retry_dec(max_attempts=4)
@timer_dec
def flaky_network_call():
    time.sleep(0.01)
    if random.random() < 0.6:
        raise ConnectionResetError("Connection dropped")
    return "Data fetched successfully"

print(flaky_network_call())

Attempt 1/4 failed: Connection dropped
[flaky_network_call] Execution took 0.0104s
Data fetched successfully


### Exercise 19: Context Manager + Generator Together — Batch File Writer
Build a context manager `BatchWriter(path)` that, on `__enter__`, opens a file and returns a generator-friendly `write_batch(lines)` method that writes multiple lines at once. On `__exit__`, it should close the file and print how many total lines were written.


In [19]:
# Your solution here

class BatchWriter:
    def __init__(self, path):
        self.path = path
        self.file = None
        self.lines_written = 0

    def __enter__(self):
        self.file = open(self.path, "w", encoding="utf-8")
        return self

    def write_batch(self, lines):
        for line in lines:
            self.file.write(f"{line}\n")
            self.lines_written += 1

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        print(f"Total lines written: {self.lines_written}")
        return False

def line_generator(count):
    for i in range(count):
        yield f"Record_{i}"

with BatchWriter("output_batch.txt") as writer:
    writer.write_batch(line_generator(5))

Total lines written: 5


### Exercise 20: Final Challenge — Rate-Limited API Call Simulator
Build a complete mini-system combining everything from today:
- A generator `simulate_api_requests(n)` that yields request IDs 1..n
- A decorator `rate_limited(seconds)` that ensures the decorated function waits at least `seconds` between calls (use `time.time()` to track the last call time)
- A context manager `APISession()` that prints "Session started" on enter and "Session closed, X requests processed" on exit (track count via an attribute)

Use all three together to process the simulated requests.


In [20]:
# Your solution here

import time
from functools import wraps

def simulate_api_requests(n):
    for req_id in range(1, n + 1):
        yield req_id

def rate_limited(seconds):
    def decorator(func):
        last_called = 0.0
        @wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal last_called
            elapsed = time.time() - last_called
            if elapsed < seconds:
                time.sleep(seconds - elapsed)
            res = func(*args, **kwargs)
            last_called = time.time()
            return res
        return wrapper
    return decorator

class APISession:
    def __init__(self):
        self.requests_processed = 0

    def __enter__(self):
        print("Session started")
        return self

    def track(self):
        self.requests_processed += 1

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Session closed, {self.requests_processed} requests processed")
        return False

@rate_limited(0.1)
def make_api_call(req_id):
    return f"Response for {req_id}"

with APISession() as session:
    for req in simulate_api_requests(4):
        result = make_api_call(req)
        session.track()
        print(f"Processed: {result}")

Session started
Processed: Response for 1
Processed: Response for 2
Processed: Response for 3
Processed: Response for 4
Session closed, 4 requests processed


---
## Lab Wrap-Up

You've now practiced:
- Writing custom iterators and making classes iterable
- Building lazy, memory-efficient generators, including pipelines and infinite sequences
- Writing decorators for logging, caching, retries, access control, and rate limiting — including decorator factories and stacking multiple decorators
- Managing resources safely with class-based and `contextlib`-based context managers
- Combining generators, decorators, and context managers together in realistic mini-systems

These patterns show up constantly in production Python code — including AI/LLM pipelines (batching data with generators, retrying API calls with decorators, and managing resources like model sessions or connections with context managers).
